# Stacking Classification - Breast Cancer Dataset


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Load Dataset

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)
target_names = data.target_names

df = X.copy()
df['Target'] = y
df['Diagnosis'] = df['Target'].map({0: 'Malignant', 1: 'Benign'})
print(df.shape)
df.head()

## EDA

In [ ]:
print(df.info())
print(df['Diagnosis'].value_counts())

In [ ]:
plt.figure(figsize=(6, 4))
df['Diagnosis'].value_counts().plot(kind='bar', color=['#7986cb', '#1a237e'])
plt.title('Diagnosis Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(X.iloc[:, :10].corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Heatmap (First 10 Features)')
plt.show()

## Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train size:', X_train.shape)
print('Test size:', X_test.shape)

## Define Base Learners and Meta Learner

In [ ]:
base_learners = [
    ('Decision Tree', DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('KNN', KNeighborsClassifier(n_neighbors=5)),
    ('SVM', SVC(probability=True, random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=50, random_state=42))
]

meta_learner = LogisticRegression(max_iter=1000, random_state=42)
print('Base Learners:', [name for name, _ in base_learners])
print('Meta Learner: Logistic Regression')

## Train Individual Models and Compare

In [ ]:
results = []
for name, clf in base_learners:
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    results.append({'Model': name, 'Accuracy': acc})
    print(f'{name}: {acc:.4f}')

## Train Stacking Model

In [ ]:
stacking_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5
)
stacking_model.fit(X_train, y_train)
y_pred = stacking_model.predict(X_test)
stacking_acc = accuracy_score(y_test, y_pred)
print('Stacking Accuracy:', stacking_acc)

## Performance Comparison

In [ ]:
results.append({'Model': 'Stacking', 'Accuracy': stacking_acc})
results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)

plt.figure(figsize=(8, 4))
colors = ['#1a237e' if m == 'Stacking' else '#7986cb' for m in results_df['Model']]
plt.bar(results_df['Model'], results_df['Accuracy'], color=colors)
plt.ylabel('Accuracy')
plt.title('Model Comparison')
plt.ylim(0.8, 1.0)
plt.show()
print(results_df)

## Evaluation

In [ ]:
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Stacking')
plt.show()

## Cross Validation

In [ ]:
cv_scores = cross_val_score(stacking_model, X, y, cv=5)
print('CV Scores:', cv_scores)
print('Mean CV Score:', cv_scores.mean())

plt.figure(figsize=(8, 4))
plt.bar(range(1, 6), cv_scores, color='#1a237e')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean: {cv_scores.mean():.2f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('5-Fold Cross Validation Scores')
plt.legend()
plt.show()